In [28]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

In [29]:
torch.cuda.is_available()

True

In [30]:
device= torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [31]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [32]:
!ls "/content/drive/MyDrive/"

 Adobe_Photoshop_CC_2019_x64.rar
'Adobe Premiere Pro 2021 v15.0.0.41 (x64) Pre-Cracked {CracksHash}.rar'
'Colab Notebooks'
 dataset
 INFO
 yunish


In [33]:
df= pd.read_csv(r"/content/drive/MyDrive/dataset/fashion-mnist.csv")

In [34]:
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [35]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test= train_test_split(df.iloc[:,1: ], df['label'], random_state= 42, test_size= 0.2)

In [36]:
scaler= StandardScaler()

scaler.fit(x_train)

x_train= scaler.transform(x_train)
x_test= scaler.transform(x_test)

In [37]:
x_train= torch.tensor(x_train, dtype= torch.float32)
x_test= torch.tensor(x_test, dtype= torch.float32)
y_train= torch.tensor(y_train.values, dtype= torch.float32)
y_test= torch.tensor(y_test.values, dtype= torch.float32)

In [38]:
from torch.utils.data import Dataset, DataLoader

In [39]:
class CustomData(Dataset):
    def __init__ (self, features, labels):
        self.features= features
        self.label= labels


    def __len__(self):
        return self.features.shape[0]


    def __getitem__(self, index):
        return self.features[index], self.label[index]

In [40]:
train_data= CustomData(x_train, y_train)

In [41]:
test_data= CustomData(x_test, y_test)

In [42]:
train_dataset= DataLoader(dataset= train_data, batch_size= 32, shuffle= True, pin_memory= True)
test_dataset= DataLoader(dataset= test_data, batch_size= 32, shuffle= False, pin_memory= True)

In [43]:
len(train_dataset) # total number of batches formed

1500

In [46]:
# designing nn
class MyNN(nn.Module):
    def __init__(self, num_features):

        super().__init__()
        self.ann= nn.Sequential(

            nn.Linear(num_features, 128), # 1st hidden layer
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(p= 0.3),   #setting dropout


            nn.Linear(128, 32),
            nn.BatchNorm1d(32),     #sevcond hidden layer
            nn.ReLU(),
            nn.Dropout(p= 0.3),


            nn.Linear(32, 10)     #output layer. We arent using softmax here because it is already applied in the loss function internally

        )


    def forward(self, data):
        return self.ann(data)

In [47]:
model= MyNN(x_train.shape[1])
model= model.to(device= device)

In [48]:
epoch= 100
learning_rate= 0.1

In [49]:
#defining optimizer
optimizer= torch.optim.SGD(params= model.parameters(), lr= learning_rate, weight_decay= 1e-4)

In [50]:
# defining loss function

critetation= nn.CrossEntropyLoss()

In [51]:
# training loop

for i in range(epoch):
    total_loss= 0

    for batch_features, batch_label in train_dataset:

        batch_features= batch_features.to(device= device)
        batch_label= batch_label.to(device= device)
        
        y_pred= model(batch_features)


        loss= critetation(y_pred, batch_label.to(dtype= torch.long))

        with torch.no_grad():
            total_loss+= loss

        #resets all the gradients
        optimizer.zero_grad()

        #calculate gradient of loss w.r.t each weight and biases
        loss.backward()



        #optimizes weights and biases
        optimizer.step()


    print(f"Epoch: {i +1}; Loss: {total_loss/len(train_dataset)}")


Epoch: 1; Loss: 0.6784507632255554
Epoch: 2; Loss: 0.5342287421226501
Epoch: 3; Loss: 0.49157530069351196
Epoch: 4; Loss: 0.4661303162574768
Epoch: 5; Loss: 0.4478742182254791
Epoch: 6; Loss: 0.43783044815063477
Epoch: 7; Loss: 0.4271499812602997
Epoch: 8; Loss: 0.4169899821281433
Epoch: 9; Loss: 0.4073536694049835
Epoch: 10; Loss: 0.40381595492362976
Epoch: 11; Loss: 0.39396199584007263
Epoch: 12; Loss: 0.39260271191596985
Epoch: 13; Loss: 0.38376906514167786
Epoch: 14; Loss: 0.38139063119888306
Epoch: 15; Loss: 0.37549465894699097
Epoch: 16; Loss: 0.37596389651298523
Epoch: 17; Loss: 0.37144702672958374
Epoch: 18; Loss: 0.36487844586372375
Epoch: 19; Loss: 0.3665589690208435
Epoch: 20; Loss: 0.3616386651992798
Epoch: 21; Loss: 0.35667768120765686
Epoch: 22; Loss: 0.35488536953926086
Epoch: 23; Loss: 0.35126790404319763
Epoch: 24; Loss: 0.3459695875644684
Epoch: 25; Loss: 0.34798887372016907
Epoch: 26; Loss: 0.3469075858592987
Epoch: 27; Loss: 0.3425464630126953
Epoch: 28; Loss: 0.342

In [52]:
model.eval()     #this says model is ready for evaluation/testing as model may behave different while training and testing

MyNN(
  (ann): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=32, bias=True)
    (5): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=32, out_features=10, bias=True)
  )
)

In [53]:
# calculate accuracy
total= 0
correct = 0
for test_batch, test_label in test_dataset:
    test_batch= test_batch.to(device= device)
    test_label= test_label.to(device= device)
    with torch.no_grad():
        y_pred= model(test_batch)

    _, idx= torch.max(y_pred, axis = 1)

    total= total + test_label.shape[0]

    correct= correct + (idx== test_label).sum()


print(correct/total)


tensor(0.8920, device='cuda:0')


In [55]:
# calculate accuracy (on training dataset)
total= 0
correct = 0
for batch_features, batch_label in train_dataset:

    batch_features= batch_features.to(device= device)
    batch_label= batch_label.to(device= device)
    with torch.no_grad():
        y_pred= model(batch_features)

    _, idx= torch.max(y_pred, axis = 1)

    total= total + test_label.shape[0]

    correct= correct + (idx==  batch_label).sum()


print(correct/total)


tensor(0.9431, device='cuda:0')


In [57]:
model.ann[4].weight

Parameter containing:
tensor([[-0.2372,  0.0641, -0.1359,  ..., -0.0345, -0.2489,  0.0546],
        [ 0.2417,  0.1937, -0.1481,  ...,  0.4276,  0.2175,  0.0216],
        [ 0.0540, -0.2664,  0.1202,  ..., -0.3884, -0.5103, -0.0738],
        ...,
        [ 0.1411,  0.1128, -0.1361,  ...,  0.0266,  0.1237, -0.1951],
        [-0.0048,  0.1272, -0.0999,  ...,  0.2562,  0.0450,  0.2968],
        [-0.0247,  0.2022, -0.0732,  ...,  0.0316, -0.0015,  0.7964]],
       device='cuda:0', requires_grad=True)